# 3D SDF -> 2D CIF + Adjacency Matrices

This notebook converts 3D SDF files to 2D CIF (multi-block CIF) and exports one adjacency matrix per molecule.

> If `rdkit` import fails in Jupyter, switch the notebook kernel/interpreter to the same Python environment used in your terminal where `rdkit` is installed.

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

INPUT_SDF_FILES = [
    "1843_actives_new.sdf",
    "Aromatase_actives_new.sdf",
    "Aromatase_inactives_new.sdf",
]

OUT_ROOT = Path("outputs_cif2d")
CIF_OUT_DIR = OUT_ROOT / "cif_2d"
ADJ_OUT_DIR = OUT_ROOT / "adjacency"
META_OUT_DIR = OUT_ROOT / "metadata"

for directory in (CIF_OUT_DIR, ADJ_OUT_DIR, META_OUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Output root:", OUT_ROOT.resolve())


In [ ]:

        def load_valid_molecules(sdf_path: Path):
            supplier = Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=True)
            return [(i, m) for i, m in enumerate(supplier) if m is not None]

        def get_molecule_name(mol: Chem.Mol, fallback_index: int) -> str:
            if mol.HasProp("_Name") and mol.GetProp("_Name").strip():
                return mol.GetProp("_Name").strip()
            if mol.HasProp("PUBCHEM_COMPOUND_CID"):
                return f"CID_{mol.GetProp('PUBCHEM_COMPOUND_CID')}"
            return f"mol_{fallback_index:06d}"

        def to_2d_mol(mol: Chem.Mol) -> Chem.Mol:
            m = Chem.Mol(mol)
            m.RemoveAllConformers()
            AllChem.Compute2DCoords(m)
            return m

        def adjacency_binary(mol: Chem.Mol) -> np.ndarray:
            n = mol.GetNumAtoms()
            mat = np.zeros((n, n), dtype=np.uint8)
            for bond in mol.GetBonds():
                i = bond.GetBeginAtomIdx()
                j = bond.GetEndAtomIdx()
                mat[i, j] = 1
                mat[j, i] = 1
            return mat

        def _bond_order_label(bond: Chem.Bond) -> str:
            bt = bond.GetBondType()
            if bt == Chem.BondType.SINGLE:
                return "S"
            if bt == Chem.BondType.DOUBLE:
                return "D"
            if bt == Chem.BondType.TRIPLE:
                return "T"
            if bt == Chem.BondType.AROMATIC:
                return "A"
            return "S"

        def mol_to_cif_block(mol: Chem.Mol, block_name: str) -> str:
            conf = mol.GetConformer()
            n = mol.GetNumAtoms()

            xs = np.array([conf.GetAtomPosition(i).x for i in range(n)], dtype=float)
            ys = np.array([conf.GetAtomPosition(i).y for i in range(n)], dtype=float)

            x_min, x_max = float(xs.min()), float(xs.max())
            y_min, y_max = float(ys.min()), float(ys.max())
            margin = 2.0

            a = max((x_max - x_min) + 2.0 * margin, 10.0)
            b = max((y_max - y_min) + 2.0 * margin, 10.0)
            c = 20.0

            x_frac = (xs - x_min + margin) / a
            y_frac = (ys - y_min + margin) / b
            z_frac = np.zeros(n, dtype=float)

            labels = []
            for i, atom in enumerate(mol.GetAtoms(), start=1):
                labels.append(f"{atom.GetSymbol()}{i}")

            lines = []
            lines.append(f"data_{block_name}")
            lines.append("_symmetry_space_group_name_H-M    'P 1'")
            lines.append(f"_cell_length_a    {a:.6f}")
            lines.append(f"_cell_length_b    {b:.6f}")
            lines.append(f"_cell_length_c    {c:.6f}")
            lines.append("_cell_angle_alpha 90")
            lines.append("_cell_angle_beta  90")
            lines.append("_cell_angle_gamma 90")
            lines.append(f"_cell_volume      {a * b * c:.6f}")
            lines.append("loop_")
            lines.append("_atom_site_label")
            lines.append("_atom_site_type_symbol")
            lines.append("_atom_site_fract_x")
            lines.append("_atom_site_fract_y")
            lines.append("_atom_site_fract_z")

            for i, atom in enumerate(mol.GetAtoms()):
                lines.append(
                    f"{labels[i]} {atom.GetSymbol()} {x_frac[i]:.6f} {y_frac[i]:.6f} {z_frac[i]:.6f}"
                )

            if mol.GetNumBonds() > 0:
                lines.append("loop_")
                lines.append("_geom_bond_atom_site_label_1")
                lines.append("_geom_bond_atom_site_label_2")
                lines.append("_geom_bond_type")
                for bond in mol.GetBonds():
                    i = bond.GetBeginAtomIdx()
                    j = bond.GetEndAtomIdx()
                    lines.append(f"{labels[i]} {labels[j]} {_bond_order_label(bond)}")

            lines.append("")
            return "
".join(lines)


In [ ]:

        summaries = []

        for sdf_name in INPUT_SDF_FILES:
            sdf_path = Path(sdf_name)
            if not sdf_path.exists():
                print(f"[SKIP] Missing file: {sdf_path}")
                continue

            mols = load_valid_molecules(sdf_path)
            if not mols:
                print(f"[SKIP] No valid molecules in: {sdf_path}")
                continue

            stem = sdf_path.stem
            out_cif = CIF_OUT_DIR / f"{stem}_2d.cif"
            out_adj = ADJ_OUT_DIR / f"{stem}_adjacency.npy"
            out_meta = META_OUT_DIR / f"{stem}_metadata.csv"

            adjacency_mats = []
            records = []

            with out_cif.open("w", encoding="utf-8", newline="
") as fh:
                for idx, mol in mols:
                    mol2d = to_2d_mol(mol)
                    name = get_molecule_name(mol2d, idx)
                    block_name = f"{stem}_{idx:06d}"

                    fh.write(mol_to_cif_block(mol2d, block_name))

                    adjacency_mats.append(adjacency_binary(mol2d))
                    records.append(
                        {
                            "molecule_index": idx,
                            "block_name": block_name,
                            "name": name,
                            "atoms": mol2d.GetNumAtoms(),
                            "bonds": mol2d.GetNumBonds(),
                        }
                    )

            np.save(out_adj, np.array(adjacency_mats, dtype=object), allow_pickle=True)
            pd.DataFrame(records).to_csv(out_meta, index=False)

            summaries.append(
                {
                    "input": str(sdf_path),
                    "molecules": len(mols),
                    "output_cif": str(out_cif),
                    "adjacency_file": str(out_adj),
                    "metadata_file": str(out_meta),
                }
            )

            print(f"[OK] {sdf_path} -> {out_cif} ({len(mols)} molecules)")

        pd.DataFrame(summaries)
